In [1]:
using Dates

include("../../src/scripts/MAVEN_load.jl")
include("../../src/scripts/MAVEN_STATIC.jl")
include("../../src/scripts/SphericalSplines.jl")
include("../../src/scripts/PhysicsCoordTransforms.jl")
include("../../src/scripts/SphericalSplinesOptim.jl")
include("../../src/scripts/MVN3DIonDisttribution.jl")
using .MAVEN_STATIC
using .MAVEN_load
using .SphericalSplines
using .PhysicsCoordTransforms
using .SphericalSplinesOptim
using .MVN3DIonDisttribution

date = Date(2022, 8, 24)
datestr = Dates.format.(date, "yyyymmdd")

ts = DateTime(date) + Dates.Hour(7) + Dates.Minute(27)
te = ts + Dates.Minute(1)

output_dir = "E:\\Output"
MAVEN_output_path = joinpath(output_dir, "MAVEN")
STATIC_output_path = joinpath(MAVEN_output_path, "STATIC")
VDF_path = joinpath(STATIC_output_path, "VDF", datestr[1:4], datestr[5:6])
mkpath(VDF_path)

mvn_mag = MVN3DIonDisttribution.read_mvn_mag(
    datestr; 
    data_root="E:/Data/MAVEN", 
    resolution=""
    )
mvn_static = MVN3DIonDisttribution.read_mvn_static_32e4d16a8m(
    datestr; 
    data_root="E:/Data/MAVEN"
    )
println("Data loaded.")

imass = 1
xd = ts + Minute.(0:1)

mvn_static_interp = MVN3DIonDisttribution.interpolate_mvn_static_eflux_3d(mvn_static, mvn_mag, xd, imass)
println("Data interpolated.")

Data loaded.
Data interpolated.


In [3]:
function nanmean(arr::AbstractArray{T,3}, dim::Int) where T
    # Count non-NaN along dim
    count_valid = sum(.!isnan.(arr), dims=dim)
    # Sum ignoring NaN
    sum_valid = sum(ifelse.(isnan.(arr), zero(T), arr), dims=dim)
    # Divide, keeping NaN where all were NaN
    result = sum_valid ./ count_valid
    result[count_valid .== 0] .= NaN
    return dropdims(result, dims=dim)
end

result = nanmean(mvn_static_interp.def, 3)


vpara = collect(range(-1000.0, 1000.0, step = 10.0))
vperp1 = collect(range(-1000.0, 1000.0, step = 10.0))

using CairoMakie
CairoMakie.activate!()

fig = Figure(size = (800, 700))
ax = Axis(fig[1, 1], 
    xlabel = "X", 
    ylabel = "Y",
    aspect = 1)  # square cells

hm = heatmap!(ax, vperp1, vpara, result,
    colormap = :gist_earth,    # same as codebase
    colorscale = log10,        # log scale (common for flux)
    colorrange = (1e4, 1e8),   # adjust to your data range
    nan_color = :transparent   # NaN shows as transparent
)

Colorbar(fig[1, 2], hm, label = "Flux")
save("heatmap.png", fig)

In [3]:
using Dates
date = Date(2022, 8, 24)
ts = DateTime(date) + Dates.Hour(7) + Dates.Minute(27)
xd = ts +Minute.(0:1)

2-element Vector{DateTime}:
 2022-08-24T07:27:00
 2022-08-24T07:28:00